<a href="https://colab.research.google.com/github/vardhan23v/agentic-ai/blob/main/MCP_With_FastMCP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP From Zero — FastMCP in Google Colab

**Goal:** Understand the MCP client-server flow with four simple calculator tools.

```text
MCP Client → MCP Server → Tools
```

This notebook is intentionally **AI-free**. We will add an LLM/agent only after the MCP plumbing is understood.

**Colab note:** We use FastMCP's **in-memory transport** so the server and client can run safely inside Colab's existing Python process. This avoids subprocess/STDIO `fileno` problems while still exercising the MCP client/server protocol.

## 1. Install FastMCP

In [2]:
%pip install -q -U fastmcp

## 2. Check FastMCP

In [3]:
import fastmcp

print("FastMCP version:", fastmcp.__version__)

FastMCP version: 4.0.10


## 3. Create the MCP Server

In [4]:
from fastmcp import FastMCP

mcp = FastMCP("Calculator Server")


@mcp.tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@mcp.tool
def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


@mcp.tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b


@mcp.tool
def divide(a: float, b: float) -> float:
    """Divide two numbers."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

print("MCP server created.")

MCP server created.


## 4. Create the MCP Client

FastMCP supports an **in-memory client** by passing the server object directly:

```python
client = Client(mcp)
```

In [5]:
from fastmcp import Client

client = Client(mcp)

print("MCP client created.")

MCP client created.


## 5. Connect the Client to the Server

In [6]:
await client.__aenter__()

print("Connected to MCP server!")

Connected to MCP server!


## 6. Discover Available Tools

In [7]:
tools = await client.list_tools()

print("Available tools:\n")

for tool in tools:
    print("-", tool.name)

Available tools:

- add
- subtract
- multiply
- divide


## 7. Understand Tool Discovery

The client asked:

```text
"What tools do you have?"
```

The server answered:

```text
add
subtract
multiply
divide
```

This is **tool discovery**.

## 8. Call the Add Tool

In [8]:
result = await client.call_tool(
    "add",
    {"a": 10, "b": 20}
)

print("Result:", result.data)

Result: 30


## 9. Call the Subtract Tool

In [9]:
result = await client.call_tool(
    "subtract",
    {"a": 20, "b": 5}
)

print("Result:", result.data)

Result: 15


## 10. Call the Multiply Tool

In [10]:
result = await client.call_tool(
    "multiply",
    {"a": 10, "b": 5}
)

print("Result:", result.data)

Result: 50


## 11. Call the Divide Tool

In [11]:
result = await client.call_tool(
    "divide",
    {"a": 20, "b": 4}
)

print("Result:", result.data)

Result: 5.0


## 12. See the MCP Flow

```text
             MCP CLIENT
                  │
                  │ tools/list
                  ▼
             MCP SERVER
                  │
          ┌───────┼────────┐
          ▼       ▼        ▼
         add     sub      mul      div
          │
          │
          ▼
        result
```

The client does not contain the calculator implementation.

The server owns the tools.

## 13. What Happens During a Tool Call?

When we write:

```python
await client.call_tool(
    "add",
    {"a": 10, "b": 20}
)
```

conceptually:

```text
Client
  │
  │ "Call add with a=10, b=20"
  ▼
MCP Server
  │
  │ executes
  ▼
add(10, 20)
  │
  ▼
30
  │
  ▼
Client
```

## 14. Who Does What?

| Component | Job |
|---|---|
| MCP Host | Application that manages MCP clients |
| MCP Client | Communicates with the MCP server |
| MCP Server | Exposes capabilities |
| Tool | Performs the actual operation |
| Agent / LLM | Later decides which tool to use |

For this lab there is **no AI agent**.

## 16. Challenge

Try these yourself:

1. Call `add(100, 250)`.
2. Call `multiply(7, 8)`.
3. Call `subtract(1000, 375)`.
4. Try dividing by zero.
5. Add a `modulo(a, b)` tool.
6. Run `list_tools()` again.
7. Call your new tool.

## 17. The Agent Comes Later

Once this is clear, add an LLM:

```text
User
 ↓
LLM / Agent
 ↓
"Which tool should I use?"
 ↓
MCP Client
 ↓
MCP Server
 ↓
Tool
 ↓
Result
 ↓
LLM / Agent
 ↓
User
```

The important separation:

**Agent = decides**

**MCP Client = communicates**

**MCP Server = exposes capabilities**

**Tool = does the work**

## 18. Cleanup

In [12]:
await client.__aexit__(None, None, None)

print("Client disconnected.")

Client disconnected.
